# Fauxtography on Colab - run everything in one go

Run this first in a **Google Colab GPU** runtime. This notebook does the
whole pipeline: environment, Drive, data split and the model training steps.

This notebook prepares a Colab session to run the fauxtography pipeline on a
GPU. It is *not* the analysis (those live in the other notebooks); it:

1. clones the repo (or reuses an already-mounted Drive copy)
2. creates a **Python 3.14** env with `uv` (Colab's own Python is 3.11/3.12,
   which our deps pin away from - see `docs/setup.md`)
3. installs the package
4. verifies the accelerator with `fauxtography env`
5. splits the dataset and runs the model commands (the GPU steps)
6. copies `results/` + checkpoints to Drive so they survive session
   disconnects

### Running this notebook: one rule to avoid stale edits

If you run this from VS Code's Colab extension, the extension syncs the
notebook you have open **and** cell 1 clones the repo (a second copy).
Editing cells while a runtime is attached edits the synced copy, but the
pipeline runs from the cloned copy - so mid-run edits can appear stale.

**Rule:** edit locally (runtime detached) -> save -> launch the runtime -> run
straight through without editing cells. A self-check cell after the clone
warns loudly if the running copy diverges from the repo's version.

Make sure you know where your dataset is BEFORE running (cell for that
below). Run cells top-to-bottom, in order.


## 1. Environment + repo

Colab runtimes persist nothing between sessions, so each session re-does this
setup. Set ``REPO_URL`` if your fork has a different name. If you'd rather use
a Drive copy, mount Drive (cell 2) and ``cd`` into it instead of cloning.


In [26]:
# Install uv (fast Python env manager that can fetch a 3.14 interpreter)
!pip install -q uv

import os
from pathlib import Path

# The repo must be PUBLIC for this clean clone to work (Settings -> General
# -> change visibility). Default branch is what the runtime fetches - it must
# be up to date (push/merge to main before running).
repo = "/content/fauxtography"
if not Path(repo).is_dir():
    !git clone --branch demetra --depth 1 \
    https://github.com/suprch4rg3d/fauxtography.git {repo}
os.chdir(repo)
print("repo ready at", os.getcwd())


repo ready at /content/fauxtography


### Self-check: is the running copy in sync with the repo?

The pipeline below runs from the clone at `{repo}`. If you edited the notebook
that the runtime synced here, the cells executing may not match what you see.
This check compares this notebook's cells to the repo's committed version and
warns loudly on any divergence.


In [28]:
import json
import subprocess
from pathlib import Path

repo = "/content/fauxtography"
this = Path("/content/notebooks/fauxtography-on-colab.ipynb")
if not this.exists():
    this = Path(repo) / "notebooks" / "fauxtography-on-colab.ipynb"

def _cells(path):
    try:
        nb = json.loads(Path(path).read_text())
    except Exception:
        return []
    return [
        "".join(c["source"])
        for c in nb["cells"]
        if c["cell_type"] == "code"
    ]

# Fetch the repo's committed notebook to compare against (no writes).
subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, capture_output=True)
committed = Path(repo) / "notebooks" / "fauxtography-on-colab.ipynb"

here = _cells(this)
comm = _cells(committed)
if here and comm and here == comm:
    print("OK - running notebook matches the repo version.")
else:
    print("WARNING: running notebook may be out of sync with the repo.")
    print("  Cells here: %d, cells in repo: %d" % (len(here), len(comm)))
    print("  If you edited cells before launching, restart the runtime and re-run.")


OK - running notebook matches the repo version.


In [29]:
%cd /content/fauxtography

!git status --short
!git branch --show-current
!git pull --ff-only origin demetra
!cat configs/clip.yaml

/content
D  results/README.md
D  results/baseline/confusion_matrix.svg
D  results/baseline/heldout_confusion_matrix.svg
D  results/baseline/heldout_metrics.json
D  results/baseline/heldout_y_pred.npy
D  results/baseline/metrics.json
D  results/baseline/y_pred.npy
R  results/clip/original-baseline/confusion_matrix.svg -> results/clip/confusion_matrix.svg
R  results/clip/original-baseline/heldout_confusion_matrix.svg -> results/clip/heldout_confusion_matrix.svg
A  results/clip/heldout_metrics.json
R  results/clip/original-baseline/heldout_y_pred.npy -> results/clip/heldout_y_pred.npy
R  results/baseline/heldout_y_true.npy -> results/clip/heldout_y_true.npy
A  results/clip/metrics.json
D  results/clip/original-baseline/heldout_metrics.json
D  results/clip/original-baseline/heldout_y_true.npy
D  results/clip/original-baseline/metrics.json
D  results/clip/original-baseline/y_true.npy
R  results/clip/original-baseline/y_pred.npy -> results/clip/y_pred.npy
R  results/baseline/y_true.npy -> re

## 2. Mount Google Drive (for dataset + persistent results)

The dataset and the outputs you want to keep (results, checkpoints) should
live on Drive, not on Colab's ephemeral disk.


In [21]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted at /content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


## 3. Create the Python 3.14 env and install the package


In [22]:
venv = "/content/fxt-venv"
!uv venv --python 3.14 {venv}
!uv pip install --python {venv}/bin/python -e .
print("package installed into", venv)


Using CPython 3.14.10
Creating virtual environment at: /content/fxt-venv
? A virtual environment already exists at `/content/fxt-venv`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt^C
Using Python 3.14.10 environment at: /content/fxt-venv
Resolved 80 packages in 355ms
Prepared 1 package in 854ms
Uninstalled 1 package in 2ms
Installed 1 package in 2ms
 ~ fauxtography==0.1.0 (from file:///content/fauxtography)
package installed into /content/fxt-venv


## 4. Verify the accelerator

This is the moment to confirm a GPU is actually attached. You should see
``cuda_available: True`` and a GPU name (e.g. Tesla T4). If you see ``cpu``,
use Runtime > Change runtime type and pick a **GPU** accelerator, then rerun.


In [23]:
!/content/fxt-venv/bin/fauxtography env


          python: 3.14.10
           torch: 2.15.0+cu132
          device: cuda
  cuda_available: True
        gpu_name: Tesla T4
   mps_available: False


## 5. Point everything at your data + Drive-backed outputs

Set these once: every command reads them via the config env-var overrides
(see `src/fauxtography/config.py`).


In [24]:
import os
venv = "/content/fxt-venv"
repo = "/content/fauxtography"
os.chdir(repo)
DATA_ROOT = "/content/drive/MyDrive/fauxtography/data"     # WHERE IS YOUR DATASET?
RUN_DIR = "/content/drive/MyDrive/fauxtography/experiments"   # checkpoints / features
RESULTS_DIR = "/content/drive/MyDrive/fauxtography/results"   # committed metrics
RAW_DIR = "/content/drive/MyDrive/fauxtography/data/raw"   # raw/ test lives here

# Export overrides - the CLI/package pick these up automatically.
os.environ["FAUXTOGRAPHY_DATA_ROOT"] = DATA_ROOT
os.environ["FAUXTOGRAPHY_RUN_DIR"] = RUN_DIR
os.environ["FAUXTOGRAPHY_RESULTS_DIR"] = RESULTS_DIR
os.environ["FAUXTOGRAPHY_HELDOUT_TEST"] = os.path.join(RAW_DIR, "test")
print("configured:", DATA_ROOT)


configured: /content/drive/MyDrive/fauxtography/data


## 6. Download the dataset into Drive (run once)

Downloads the GenImage slice configured in `configs/genimage.yaml` straight
from HuggingFace into Drive - no need to upload gigabytes from your machine.
Re-running is safe; skip this section if the data is already in Drive.

Output layout (see `docs/dataset_research.md`):
    <RAW_DIR>/train/{real,fake}          training pool
    <RAW_DIR>/test/{real,fake}           official held-out test
    <RAW_DIR>/cross/<gen>/{real,fake}    cross-generator sets


In [ ]:
import os
venv = "/content/fxt-venv"
repo = "/content/fauxtography"
os.chdir(repo)

RAW_DIR = "/content/drive/MyDrive/fauxtography/data/raw"   # where raw/ goes
os.environ["FAUXTOGRAPHY_RAW_DIR"] = RAW_DIR

# Skip the download only if BOTH the training pool and the held-out test
# are already complete. A partial/empty leaf (e.g. an interrupted download)
# would otherwise pass the old train-only check and stay broken forever.
def _has_images(*subs: str) -> bool:
    p = os.path.join(RAW_DIR, *subs)
    return os.path.isdir(p) and bool(os.listdir(p))

complete = _has_images("train", "real") and _has_images("train", "fake") and _has_images("test", "real") and _has_images("test", "fake")
if complete:
    print("raw train + held-out test already present - skipping download")
else:
    print("raw data incomplete - (re)fetching GenImage slice")
    !{venv}/bin/fauxtography fetch-genimage
print("raw data at:", RAW_DIR)


raw train + held-out test already present - skipping download
raw data at: /content/drive/MyDrive/fauxtography/data/raw


## 7. Split the dataset into train/val/test from the downloaded raw pool

Builds `data/train|val|test` from the GenImage SD v1.4 *training pool* only
(`raw/train`), keeping the official held-out `raw/test` separate for later
evaluation (train-baseline scores it when `data.heldout_test` is set).


In [ ]:
# NOTE: splits only raw/train - the official test stays untouched.
!/content/fxt-venv/bin/fauxtography prepare-data \
    --source-real /content/drive/MyDrive/fauxtography/data/raw/train/real \
    --source-fake /content/drive/MyDrive/fauxtography/data/raw/train/fake \
    --data-root /content/drive/MyDrive/fauxtography/data


[2026-11-12 14:26:17,928] INFO fauxtography.cli: class=real files=10000 -> train=7000 val=1500 test=1500
[2026-11-12 14:36:50,691] INFO fauxtography.cli: class=fake files=10000 -> train=7000 val=1500 test=1500
[2026-11-12 15:39:49,031] INFO fauxtography.cli: Done. Structure written under /content/drive/MyDrive/fauxtography/data


## 8. Training / evaluation (GPU steps - run one at a time)

The three model commands are heavy; run them in separate cells so a disconnect
can't lose more than one step. Each reads its own config and writes results to
Drive via the env vars above.


In [ ]:
# Baseline - fine-tune the shared ResNet (joint)
!/content/fxt-venv/bin/fauxtography train-baseline


Traceback (most recent call last):
  File "/content/fxt-venv/bin/fauxtography", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/fauxtography/src/fauxtography/cli.py", line 626, in main
    return args.func(args)
           ~~~~~~~~~^^^^^^
  File "/content/fauxtography/src/fauxtography/cli.py", line 99, in cmd_train_baseline
    from .data.dataset import FauxtoDataset
  File "/content/fauxtography/src/fauxtography/data/__init__.py", line 3, in <module>
    from .dataset import CLASS_NAMES, FauxtoDataset, get_datasets
  File "/content/fauxtography/src/fauxtography/data/dataset.py", line 21, in <module>
    from torchvision.datasets import ImageFolder
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1333, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1267, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1296, in find_spec
  File "<frozen importlib._

In [25]:
%cd /content/fauxtography

!git branch --show-current
!git log -1 --oneline
!cat configs/clip.yaml

import sys
from pathlib import Path

# Add the project's source directory to Colab's Python path
sys.path.insert(0, "/content/fauxtography/src")

from fauxtography.config import load_config

config_path = Path("/content/fauxtography/configs/clip.yaml")
cfg = load_config("clip", config_path)

print("REAL PROMPTS:")
for prompt in cfg.prompts_real:
    print("-", prompt)

print("\nFAKE PROMPTS:")
for prompt in cfg.prompts_fake:
    print("-", prompt)

/content
demetra
bb72c69 (grafted, HEAD -> demetra, origin/demetra) Created using Colab
# CLIP zero-shot
model_name: openai/clip-vit-base-patch32
batch_size: 64
num_workers: 4

# Original baseline
# prompts_real:
#   - "a real photograph taken with a camera"
#   - "a genuine photo of a real scene"

# prompts_fake:
#   - "an image generated by artificial intelligence"
#   - "a fake AI-generated image"

# Minimal description
prompts_real:
  - "a real image"
  - "a photograph"

prompts_fake:
  - "a fake image"
  - "an AI-generated image"

# # Balanced descriptions
# prompts_real:
#   - "an image of a scene captured by a real camera"
#   - "a photograph of a scene created by photographic capture"

# prompts_fake:
#   - "an image of a scene generated by an AI model"
#   - "a photograph-like image of a scene created by artificial intelligence"

# # Photographic characteristics
# prompts_real:
#   - "a natural photograph with realistic lighting, shadows, and textures"
#   - "a camera photogra

ModuleNotFoundError: No module named 'fauxtography.config'

In [12]:

# CLIP zero-shot - DemetraP20. NOT IMPLEMENTED YET.
# Uncomment (remove the leading #) once eval-clip is implemented.
!/content/fxt-venv/bin/fauxtography eval-clip
# print("SKIPPED: CLIP eval-clip not implemented yet (see docs/milestones).")


[2026-11-13 10:10:36,781] INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/revision/main "HTTP/1.1 200 OK"
[2026-11-13 10:10:36,888] INFO httpx2: HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
[2026-11-13 10:10:37,000] INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32 "HTTP/1.1 200 OK"
[2026-11-13 10:10:37,137] INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/commits/main "HTTP/1.1 200 OK"

[2026-11-13 10:10:37,282] INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/discussions?p=0 "HTTP/1.1 200 OK"
[2026-11-13 10:10:37,388] INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/commits/refs%2Fpr%2F66 "HTTP/1.1 200 OK"
[2026-11-13 10:10:37,505] INFO httpx2: HTTP Request: HEAD https://huggingface

In [ ]:
# DINOv2 - extract features then train the linear probe - suprch4rg3d.
!/content/fxt-venv/bin/fauxtography extract-dinov2
!/content/fxt-venv/bin/fauxtography train-linear-probe


Traceback (most recent call last):
  File "/content/fxt-venv/bin/fauxtography", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/fauxtography/src/fauxtography/cli.py", line 626, in main
    return args.func(args)
           ~~~~~~~~~^^^^^^
  File "/content/fauxtography/src/fauxtography/cli.py", line 314, in cmd_extract_dinov2
    from .data.dataset import FauxtoDataset, get_datasets
  File "/content/fauxtography/src/fauxtography/data/__init__.py", line 3, in <module>
    from .dataset import CLASS_NAMES, FauxtoDataset, get_datasets
  File "/content/fauxtography/src/fauxtography/data/dataset.py", line 21, in <module>
    from torchvision.datasets import ImageFolder
  File "/content/fxt-venv/lib/python3.14/site-packages/torchvision/__init__.py", line 8, in <module>
    from torchvision import (
    ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )  # usort:skip
    ^
  File "/content/fxt-venv/lib/python3.14/site-packages/torchvision/datasets/__init__.py", 

## 9. Backup sanity

Confirm the outputs actually landed on Drive (they survive session drops).
Then open `00_data_overview.ipynb` and the model analysis notebooks to
inspect the results and write the verdicts.


In [13]:
!ls /content/drive/MyDrive/fauxtography/results
!echo "---" && ls /content/drive/MyDrive/fauxtography/results/clip 2>/dev/null || echo "no clip results yet"


clip
---
confusion_matrix.svg	      heldout_y_pred.npy  y_pred.npy
heldout_confusion_matrix.svg  heldout_y_true.npy  y_true.npy
heldout_metrics.json	      metrics.json


## 10. Publish results to git

Colab runs on your branch - these cells copy the results from Drive into the
repo and **stage only `results/`** (the small committed artifacts: json, svg,
csv). Checkpoints and features are already git-ignored, so nothing heavy gets
staged.

Deliberately *not* automated: review what's staged, then commit and push
**from your own branch** and open a PR into `dev` (see `docs/work_division.md`).
No credentials are handled here - set up GitHub auth on the machine/terminal
that pushes (e.g. a PAT scoped to this repo).


In [14]:
# Bring the results (on Drive) into the cloned repo so git sees them
import shutil
shutil.rmtree("/content/fauxtography/results", ignore_errors=True)
shutil.copytree("/content/drive/MyDrive/fauxtography/results", "/content/fauxtography/results")
# Stage only results/ (checkpoints + features stay out; they are gitignored)
!cd /content/fauxtography && git add results/
!cd /content/fauxtography && git status --short results/
print("review the staged list, then commit + push from your branch")


D  results/README.md
D  results/baseline/confusion_matrix.svg
D  results/baseline/heldout_confusion_matrix.svg
D  results/baseline/heldout_metrics.json
D  results/baseline/heldout_y_pred.npy
D  results/baseline/metrics.json
D  results/baseline/y_pred.npy
R  results/clip/original-baseline/confusion_matrix.svg -> results/clip/confusion_matrix.svg
R  results/clip/original-baseline/heldout_confusion_matrix.svg -> results/clip/heldout_confusion_matrix.svg
A  results/clip/heldout_metrics.json
R  results/clip/original-baseline/heldout_y_pred.npy -> results/clip/heldout_y_pred.npy
R  results/baseline/heldout_y_true.npy -> results/clip/heldout_y_true.npy
A  results/clip/metrics.json
D  results/clip/original-baseline/heldout_metrics.json
D  results/clip/original-baseline/heldout_y_true.npy
D  results/clip/original-baseline/metrics.json
D  results/clip/original-baseline/y_true.npy
R  results/clip/original-baseline/y_pred.npy -> results/clip/y_pred.npy
R  results/baseline/y_true.npy -> results/cli

In [15]:
# ---- do this on your branch / terminal, NOT automatically here ----
#   cd /content/fauxtography
#   git add results/
#   git commit -m "Add <model> results"
#   git push origin <your-branch>
#   # then open a PR: <your-branch> -> dev
print("manual step - see cells above")


manual step - see cells above
